# 00 - Download STJ Metadados

Objetivo: baixar iterativamente apenas os arquivos `metadados*.json` da base STJ - Integras de Decisoes Terminativas e Acordaos, sem baixar os ZIPs de texto.

Os arquivos sao salvos em subpastas por ano, por exemplo `raw/stj_integras_metadata/2026/`. Isso facilita ampliar ou restringir o recorte temporal nos notebooks seguintes.


## 1. Ambiente

In [ ]:
# Rode no Colab se necessario.
# !pip install requests beautifulsoup4 tqdm pandas

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, unquote
import re
import time
import random

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [ ]:
from google.colab import drive

MOUNT_POINT = Path('/content/drive')

if (MOUNT_POINT / 'MyDrive').exists():
    print('Google Drive ja esta montado.')
else:
    drive.mount(str(MOUNT_POINT), force_remount=True)

DRIVE_DATA = Path('/content/drive/MyDrive/Mestrado/2026/llms/data')
METADATA_RAW = DRIVE_DATA / 'raw' / 'stj_integras_metadata'
REPORTS_DATA = DRIVE_DATA / 'reports' / 'summaries'

for path in [METADATA_RAW, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print('METADATA_RAW:', METADATA_RAW)

## 2. Descobrir links de metadados

A pagina lista recursos JSON e ZIP. Aqui filtramos apenas URLs cujo nome final segue exatamente o padrao `metadadosAAAAMMDD.json` ou `metadadosAAAAMM.json`.

In [ ]:
DATASET_URL = 'https://dadosabertos.web.stj.jus.br/dataset/integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 datajud-probe academic metadata downloader',
}

response = requests.get(DATASET_URL, headers=HEADERS, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

metadata_urls = []
skipped_metadata_like = []

for a in soup.select('a[href]'):
    href = urljoin(DATASET_URL, a['href'])
    filename = unquote(href.split('/')[-1])

    if filename.startswith('metadados') and filename.endswith('.jso'):
        skipped_metadata_like.append(href)
        continue

    # O portal alterna entre URLs como metadados20240112.json e metadados20240205.
    # Aceitamos os dois e normalizamos o nome local com .json no download.
    if re.fullmatch(r'metadados\d{6,8}(?:\.json)?', filename):
        metadata_urls.append(href)

metadata_urls = sorted(set(metadata_urls))
skipped_metadata_like = sorted(set(skipped_metadata_like))

print('Arquivos de metadados .json encontrados:', len(metadata_urls))
print('Links parecidos, mas ignorados:', len(skipped_metadata_like))
print('Primeiros:', [unquote(url.split('/')[-1]) for url in metadata_urls[:5]])
print('Ultimos:', [unquote(url.split('/')[-1]) for url in metadata_urls[-5:]])

## 3. Selecionar recorte

Comece com um recorte. Baixar todos os arquivos de uma vez pode pressionar o servidor e aumentar a chance de erros 5xx/522.

In [ ]:
# Ajuste o recorte aqui.
# Use None para nao limitar um lado do intervalo.
DATA_INICIO = '2026-01-01'
DATA_FIM = '2026-12-31'

# Para um teste pequeno, defina um numero. Para baixar todo o recorte selecionado, deixe None.
MAX_DOWNLOADS = None

def metadata_filename_from_url(url: str) -> str:
    filename = unquote(url.split('/')[-1].split('?')[0])
    match = re.fullmatch(r'(metadados\d{6,8})(?:\.json)?', filename)
    return f'{match.group(1)}.json' if match else filename


def date_from_metadata_filename(filename: str) -> pd.Timestamp | None:
    match = re.fullmatch(r'metadados(\d{6}|\d{8})(?:\.json)?', filename)
    if not match:
        return None
    raw_date = match.group(1)
    fmt = '%Y%m%d' if len(raw_date) == 8 else '%Y%m'
    return pd.to_datetime(raw_date, format=fmt, errors='coerce')


def date_from_metadata_url(url: str) -> pd.Timestamp | None:
    return date_from_metadata_filename(metadata_filename_from_url(url))

start = pd.to_datetime(DATA_INICIO) if DATA_INICIO else None
end = pd.to_datetime(DATA_FIM) if DATA_FIM else None

selected_urls = []
for url in metadata_urls:
    url_date = date_from_metadata_url(url)
    if url_date is None or pd.isna(url_date):
        continue
    if start is not None and url_date < start:
        continue
    if end is not None and url_date > end:
        continue
    selected_urls.append(url)

selected_urls = sorted(selected_urls, key=date_from_metadata_url)

if MAX_DOWNLOADS is not None:
    selected_urls = selected_urls[:MAX_DOWNLOADS]

print('Selecionados:', len(selected_urls))
print([unquote(url.split('/')[-1]) for url in selected_urls[:10]])

## 4. Download robusto

Erros 5xx e 522 costumam ser temporarios. A funcao abaixo tenta novamente com espera crescente. Se ainda falhar, registra a falha e segue para o proximo arquivo.

In [ ]:
def download_metadata_file(
    url: str,
    output_dir: Path,
    overwrite: bool = False,
    attempts: int = 4,
    base_sleep: float = 3.0,
) -> dict:
    filename = metadata_filename_from_url(url)
    file_date = date_from_metadata_filename(filename)
    year_dir = output_dir / str(file_date.year) if file_date is not None and not pd.isna(file_date) else output_dir / 'sem_data'
    year_dir.mkdir(parents=True, exist_ok=True)
    output_path = year_dir / filename

    if output_path.exists() and not overwrite:
        return {
            'url': url,
            'filename': filename,
            'status': 'already_exists',
            'path': str(output_path),
            'error': None,
        }

    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            with requests.get(url, headers=HEADERS, stream=True, timeout=180) as response:
                response.raise_for_status()
                tmp_path = output_path.with_suffix(output_path.suffix + '.part')

                with tmp_path.open('wb') as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

                tmp_path.replace(output_path)

            return {
                'url': url,
                'filename': filename,
                'status': 'downloaded',
                'path': str(output_path),
                'error': None,
            }
        except requests.HTTPError as exc:
            status_code = exc.response.status_code if exc.response is not None else None
            last_error = f'HTTP {status_code}: {exc}'
        except requests.RequestException as exc:
            last_error = repr(exc)

        sleep_seconds = base_sleep * attempt + random.uniform(0, 2)
        print(f'Falha em {filename} tentativa {attempt}/{attempts}: {last_error}. Aguardando {sleep_seconds:.1f}s')
        time.sleep(sleep_seconds)

    return {
        'url': url,
        'filename': filename,
        'status': 'failed',
        'path': str(output_path),
        'error': last_error,
    }

In [ ]:
results = []

for url in tqdm(selected_urls):
    result = download_metadata_file(url, METADATA_RAW, overwrite=False)
    results.append(result)

    # Pausa leve entre arquivos para nao martelar o portal.
    time.sleep(random.uniform(0.8, 2.0))

download_report = pd.DataFrame(results)
download_report_path = REPORTS_DATA / 'stj_metadata_download_report.csv'
download_report.to_csv(download_report_path, index=False)

print(download_report['status'].value_counts(dropna=False))
print('Relatorio:', download_report_path)
download_report.tail()

## 5. Conferencia local

In [ ]:
local_metadata_files = sorted(METADATA_RAW.rglob('metadados*.json'))
local_inventory = pd.DataFrame({
    'arquivo': [p.name for p in local_metadata_files],
    'pasta': [p.parent.name for p in local_metadata_files],
})

print('Arquivos JSON locais:', len(local_metadata_files))
if not local_inventory.empty:
    print(local_inventory.groupby('pasta').size().rename('arquivos').reset_index())
print('Primeiros:', [p.name for p in local_metadata_files[:5]])
print('Ultimos:', [p.name for p in local_metadata_files[-5:]])


## 5.1. Opcional: organizar arquivos soltos por ano

Se voce ja baixou arquivos diretamente em `stj_integras_metadata/`, rode esta celula uma vez para mover cada `metadados*.json` para a subpasta do ano correspondente.


In [ ]:
loose_files = sorted([p for p in METADATA_RAW.glob("metadados*.json") if p.is_file()])
print("Arquivos soltos encontrados:", len(loose_files))

for path in loose_files:
    file_date = date_from_metadata_filename(path.name)
    target_dir = METADATA_RAW / str(file_date.year) if file_date is not None and not pd.isna(file_date) else METADATA_RAW / "sem_data"
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / path.name

    if target_path.exists():
        print("Ja existe, removendo solto:", path.name)
        path.unlink()
    else:
        path.replace(target_path)
        print("Movido:", path.name, "->", target_dir.name)


## 6. Reexecutar falhas

Se houver falhas, aguarde alguns minutos e rode esta celula para tentar apenas os arquivos que falharam.

In [ ]:
failed_urls = download_report.loc[download_report['status'].eq('failed'), 'url'].tolist() if 'download_report' in globals() else []
print('Falhas para reexecutar:', len(failed_urls))

retry_results = []
for url in tqdm(failed_urls):
    retry_results.append(download_metadata_file(url, METADATA_RAW, overwrite=False, attempts=6, base_sleep=10.0))
    time.sleep(random.uniform(3.0, 6.0))

retry_report = pd.DataFrame(retry_results)
if not retry_report.empty:
    retry_report_path = REPORTS_DATA / 'stj_metadata_download_retry_report.csv'
    retry_report.to_csv(retry_report_path, index=False)
    print(retry_report['status'].value_counts(dropna=False))
    print('Relatorio retry:', retry_report_path)

retry_report